# JointGuardian — Phase 1: C-MAPSS LSTM Training (Kaggle)

Run these cells **in order, top to bottom**. Before running Cell 5, go to
**Settings → Accelerator → GPU T4 x2** in the right sidebar, or training
will fall back to CPU.

Two placeholders you must edit before running:
- **Cell 2**: `REPO_URL` — your GitHub repo URL (use a token in the URL if the repo is private)
- **Cell 3**: `DATA_INPUT_DIR` — the name of your Kaggle Dataset input containing train_FD001.txt / test_FD001.txt / RUL_FD001.txt


### Cell 1

Environment setup — installs anything Kaggle doesn't already ship with.

In [ ]:
import sys, os, subprocess

# Kaggle ships torch, pandas, numpy, matplotlib pre-installed.
# Install the few extras JointGuardian needs that may be missing.
subprocess.run(["pip", "install", "-q", "pyyaml"], check=False)

print("Environment ready.")


### Cell 2

Clone the JointGuardian repo from GitHub and make src/ importable.

In [ ]:
REPO_URL = "https://github.com/<your-username>/joint-guardian.git"  # <-- EDIT THIS
REPO_DIR = "/kaggle/working/joint-guardian"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
sys.path.append(os.path.join(REPO_DIR, "src"))
print("Working directory:", os.getcwd())


### Cell 3

Copy the C-MAPSS data files from your Kaggle Dataset input into data/raw/.

In [ ]:
import shutil

DATA_INPUT_DIR = "/kaggle/input/<your-cmapss-dataset-name>"  # <-- EDIT THIS
os.makedirs("data/raw", exist_ok=True)

for fname in ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt"]:
    src_path = os.path.join(DATA_INPUT_DIR, fname)
    dst_path = os.path.join("data/raw", fname)
    shutil.copy(src_path, dst_path)
    print(f"Copied {fname}")


### Cell 4

Confirm the GPU is actually enabled before training.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — enable it under Settings > Accelerator > GPU T4 x2, then re-run this cell.")


### Cell 5

Run Phase 1 training. Uses configs/cmapss_config.yaml for all hyperparameters.

In [ ]:
!python src/training/train_cmapss.py


### Cell 6

Run Phase 1 evaluation — this is the verification checkpoint (pred-vs-actual scatter + RMSE/NASA score).

In [ ]:
!python scripts/evaluate_cmapss.py


### Cell 7

Display the verification checkpoint plot inline.

In [ ]:
from IPython.display import Image
Image(filename="outputs/plots/phase1_pred_vs_actual.png")


### Cell 8

Render a training-progress **video** — the loss curve draws itself in over
epochs, like a training timelapse, so you have something shareable beyond a
static plot.

**Note:** this assumes `train_cmapss.py` saved loss history to
`outputs/plots/loss_history.json` as `{"train_loss": [...], "val_loss": [...]}`.
Check the actual key names your coding agent used (Prompt 5 allowed JSON or
CSV) and adjust the load step below if they differ.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.animation as animation

os.makedirs("outputs/videos", exist_ok=True)

with open("outputs/plots/loss_history.json") as f:
    history = json.load(f)

train_loss = history["train_loss"]
val_loss = history["val_loss"]
epochs = list(range(1, len(train_loss) + 1))

fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_color("#333333")
ax.set_xlabel("Epoch", color="white")
ax.set_ylabel("Loss (MSE)", color="white")
ax.set_title("JointGuardian — Phase 1 LSTM Training", color="white", fontsize=14)

line_train, = ax.plot([], [], color="#2a78d6", linewidth=2, label="Train loss")
line_val, = ax.plot([], [], color="#e0623c", linewidth=2, label="Validation loss")
epoch_text = ax.text(0.02, 0.95, "", transform=ax.transAxes, color="white",
                      fontsize=12, va="top")
legend = ax.legend(facecolor="#0d1117", loc="upper right")
for text in legend.get_texts():
    text.set_color("white")

ax.set_xlim(1, len(epochs))
ax.set_ylim(0, max(train_loss + val_loss) * 1.1)

def update(frame):
    line_train.set_data(epochs[:frame + 1], train_loss[:frame + 1])
    line_val.set_data(epochs[:frame + 1], val_loss[:frame + 1])
    epoch_text.set_text(
        f"Epoch {frame + 1}/{len(epochs)}\n"
        f"Train loss: {train_loss[frame]:.4f}\n"
        f"Val loss: {val_loss[frame]:.4f}"
    )
    return line_train, line_val, epoch_text

anim = animation.FuncAnimation(fig, update, frames=len(epochs), interval=150, blit=False)

try:
    writer = animation.FFMpegWriter(fps=10, bitrate=1800)
    anim.save("outputs/videos/phase1_training_report.mp4", writer=writer)
    print("Saved outputs/videos/phase1_training_report.mp4")
except Exception as e:
    print("FFmpeg unavailable, saving as GIF instead:", e)
    anim.save("outputs/videos/phase1_training_report.gif", writer=animation.PillowWriter(fps=10))
    print("Saved outputs/videos/phase1_training_report.gif")

plt.close(fig)


### Cell 9

Save one combined still image (loss curve + scatter side by side) — good for a single shareable post.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor("white")

axes[0].plot(epochs, train_loss, label="Train loss", color="#2a78d6")
axes[0].plot(epochs, val_loss, label="Validation loss", color="#e0623c")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (MSE)")
axes[0].set_title("Training curve")
axes[0].legend()

scatter_img = plt.imread("outputs/plots/phase1_pred_vs_actual.png")
axes[1].imshow(scatter_img)
axes[1].axis("off")
axes[1].set_title("Predicted vs actual RUL")

fig.suptitle("JointGuardian — Phase 1 Training Report", fontsize=15)
fig.tight_layout()
fig.savefig("outputs/plots/phase1_training_report.png", dpi=150, facecolor="white")
plt.close(fig)
print("Saved outputs/plots/phase1_training_report.png")


### Cell 10

Package the trained model, plots, and video so they're easy to download from the Kaggle Output panel.

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/joint_guardian_phase1_outputs", "zip", "outputs")
shutil.copy("models/lstm_pdm.pth", "/kaggle/working/lstm_pdm.pth")

print("Ready to download from the Output panel:")
print(" - joint_guardian_phase1_outputs.zip")
print(" - lstm_pdm.pth")
